# 02 — Stabl Feature Selection

**Pipeline Step 2 of 5**

Load preprocessed AnnData, run Stabl stability-based feature selection
(L1-penalised logistic regression, 500 bootstraps), and cache results.

### Inputs
- `data/processed/mouse_brain_preprocessed.h5ad`

### Outputs
- `cache/stabl_results_<hash>.pkl`
- `cache/stabl_features_<hash>.csv`

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.spatial_pipeline import load_adata, run_stabl_cached

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
CACHE_DIR = PROJECT_ROOT / "cache"

print("Imports ready.")

Imports ready.


## 2.1 Load Preprocessed Data

In [2]:
adata = load_adata(DATA_PROCESSED / "mouse_brain_preprocessed.h5ad")
print(f"\nLoaded: {adata.shape[0]} spots × {adata.shape[1]} genes")

  Loading dataset: /Users/shaunfchen/Documents/Repositories/spatial-microckg-agent/data/processed/mouse_brain_preprocessed.h5ad
  Shape: 2691 spots × 19652 genes

Loaded: 2691 spots × 19652 genes


## 2.2 Run Stabl Feature Selection

1. Select 2,000 HVGs
2. Compute Leiden clusters for binary label assignment
3. Run Stabl (500 bootstraps)
4. FDP+ threshold determined automatically

Cached results load in < 1 second.

In [3]:
stabl_result = run_stabl_cached(
    adata,
    cache_dir=CACHE_DIR,
    dataset_name="squidpy",
    n_hvgs=2000,
    label_method="cluster",
    n_bootstraps=500,
)

print(f"\nStabl results:")
print(f"  Features selected: {stabl_result['n_selected']}")
print(f"  FDP+ threshold: {stabl_result['threshold']:.4f}")
print(f"  Minimum FDP+: {stabl_result['fdr']:.4f}")

  Loading cached Stabl results: /Users/shaunfchen/Documents/Repositories/spatial-microckg-agent/cache/stabl_results_eba9686a7ee7.pkl

Stabl results:
  Features selected: 41
  FDP+ threshold: 0.8700
  Minimum FDP+: 0.0244


## 2.3 Review Selected Features

In [4]:
import pandas as pd

df_features = pd.DataFrame({
    "Gene": stabl_result["selected_genes"],
    "Stability Score": [
        stabl_result["stability_scores"][g]
        for g in stabl_result["selected_genes"]
    ],
}).sort_values("Stability Score", ascending=False).reset_index(drop=True)

print(f"\nTop Stabl-certified biomarkers ({len(df_features)} total):")
df_features.head(20)


Top Stabl-certified biomarkers (41 total):


,Gene,Stability Score
0,Kcnip2,1.000
1,Wnt4,1.000
2,Pcdh8,1.000
3,Cplx2,1.000
4,Stxbp6,1.000
5,Ctxn3,1.000
6,Lpl,1.000
7,Lmo3,1.000
8,Penk,1.000
9,Lamp5,1.000


In [5]:
print(f"Score statistics:")
print(f"  Mean:   {df_features['Stability Score'].mean():.4f}")
print(f"  Median: {df_features['Stability Score'].median():.4f}")
print(f"  Min:    {df_features['Stability Score'].min():.4f}")
print(f"  Max:    {df_features['Stability Score'].max():.4f}")
print(f"  Perfect (1.0): {(df_features['Stability Score'] == 1.0).sum()}")

Score statistics:
  Mean:   0.9550
  Median: 0.9700
  Min:    0.8720
  Max:    1.0000
  Perfect (1.0): 10
